# Preprocessing

In [2]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# from category_encoders import TargetEncoder # Optional: for high-cardinality

import sys
sys.path.append(os.path.abspath(os.path.join('..')))
from src import config

print("✅ Setup complete.")

✅ Setup complete.


In [3]:
# Generic loading mechanism
INTERIM_DATA_PATH = '../data/interim/water_quality_mvp_baseline.parquet' 
df = pd.read_parquet(INTERIM_DATA_PATH)

print(f"Loaded shape: {df.shape}")
df.head()

Loaded shape: (9319, 16)


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,Elevation_m,soil_ph,soil_clay_g_kg,nir,green,swir16,swir22,NDMI,MNDWI,pet
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,173.0,8.2,69.0,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595,174.2
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,1524.0,6.5,302.0,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,124.1
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,1472.0,6.1,260.0,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,127.5
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,1342.0,6.8,264.0,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,129.7
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,1356.0,6.7,245.0,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,129.2


### Time Extraction & Redundancy Pruning

In [4]:
# --- Time Extraction ---
# Convert Sample Date to datetime, extract month as a categorical string
df['Sample Date'] = pd.to_datetime(df['Sample Date'], errors='coerce', dayfirst=True)
df['Month'] = df['Sample Date'].dt.month.astype(str)

# Drop the original date column (not useful for tree models as-is)
# Also drop Latitude/Longitude (used for data collection, not direct features here)
df = df.drop(columns=['Sample Date', 'Latitude', 'Longitude'])

# --- Redundancy Pruning ---
# Drop swir22 to reduce multicollinearity (highly correlated with swir16/nir)
df = df.drop(columns=['swir22'])

print(f"Shape after feature extraction & pruning: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape after feature extraction & pruning: (9319, 13)
Columns: ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Elevation_m', 'soil_ph', 'soil_clay_g_kg', 'nir', 'green', 'swir16', 'NDMI', 'MNDWI', 'pet', 'Month']


### Train/Test Split

In [5]:
TARGET_COL = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

X = df.drop(columns=TARGET_COL)
y = df[TARGET_COL]

print(f"Features: {X.columns.tolist()}")
print(f"Targets:  {y.columns.tolist()}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

Features: ['Elevation_m', 'soil_ph', 'soil_clay_g_kg', 'nir', 'green', 'swir16', 'NDMI', 'MNDWI', 'pet', 'Month']
Targets:  ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
X shape: (9319, 10), y shape: (9319, 3)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print(f"Training: X={X_train.shape}, y={y_train.shape}")
print(f"Testing:  X={X_test.shape}, y={y_test.shape}")

Training: X=(7455, 10), y=(7455, 3)
Testing:  X=(1864, 10), y=(1864, 3)


### Feature Grouping

In [7]:
# Automatically categorize columns
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Note: In a real project, manually move high-cardinality or ordinal columns 
# to separate lists here if they require different treatment (e.g., TargetEncoding)

print(f"Numeric features ({len(num_cols)}): {num_cols[:5]}...")
print(f"Categorical features ({len(cat_cols)}): {cat_cols[:5]}...")

Numeric features (9): ['Elevation_m', 'soil_ph', 'soil_clay_g_kg', 'nir', 'green']...
Categorical features (1): ['Month']...


### Building Preprocessor

In [12]:
# --- Numeric Pipeline ---
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
])

# --- Categorical Pipeline ---
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# --- Combine ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols),
    ],
    remainder='drop',
)

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


### Fit and Transform

In [9]:
# Fit on training data ONLY, then transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"Resulting feature count: {len(feature_names)}")
print(f"Feature names: {feature_names.tolist()}")

Resulting feature count: 29
Feature names: ['num__Elevation_m', 'num__soil_ph', 'num__soil_clay_g_kg', 'num__nir', 'num__green', 'num__swir16', 'num__NDMI', 'num__MNDWI', 'num__pet', 'num__missingindicator_Elevation_m', 'num__missingindicator_soil_ph', 'num__missingindicator_soil_clay_g_kg', 'num__missingindicator_nir', 'num__missingindicator_green', 'num__missingindicator_swir16', 'num__missingindicator_NDMI', 'num__missingindicator_MNDWI', 'cat__Month_1', 'cat__Month_10', 'cat__Month_11', 'cat__Month_12', 'cat__Month_2', 'cat__Month_3', 'cat__Month_4', 'cat__Month_5', 'cat__Month_6', 'cat__Month_7', 'cat__Month_8', 'cat__Month_9']


### Save

In [13]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Rebuild DataFrames with proper column names
train_fe = pd.DataFrame(X_train_processed, columns=feature_names)
for col in TARGET_COL:
    train_fe[col] = y_train[col].values

test_fe = pd.DataFrame(X_test_processed, columns=feature_names)
for col in TARGET_COL:
    test_fe[col] = y_test[col].values

# Save as Parquet (preserves dtypes, faster, smaller than CSV)
train_fe.to_parquet('../data/processed/train_fe.parquet', index=False)
test_fe.to_parquet('../data/processed/test_fe.parquet', index=False)

# Save the fitted preprocessor pipeline
PIPELINE_PATH = '../models/preprocessor.joblib'
joblib.dump(preprocessor, PIPELINE_PATH)

print(f"Train set: {train_fe.shape}")
print(f"Test set:  {test_fe.shape}")
print(f"Pipeline saved to: {PIPELINE_PATH}")
print("Preprocessing complete.")

Train set: (7455, 32)
Test set:  (1864, 32)
Pipeline saved to: ../models/preprocessor.joblib
Preprocessing complete.
